<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# The Linear Regression Engine
## Dataset: Artisan Cheese Fermentation Time Prediction

**Project:** 001 — The AI Engineering Lab  
**Objective:** Build a complete linear regression pipeline from first principles, covering Ordinary Least Squares (OLS), Ridge (L2), Lasso (L1), and Elastic Net regularization.

**What you will learn:**
- How OLS finds the best-fit line using the closed-form Normal Equation
- How Gradient Descent iteratively minimizes the cost function
- Why regularization prevents overfitting and what L1 vs L2 actually do geometrically
- How to choose the right regularization strength using cross-validation
- How to properly evaluate regression models and interpret residuals

**Dataset:** Predicting the optimal fermentation time (in hours) of artisan cheese based on biochemical and environmental factors. This is a synthetic dataset designed to mimic real-world fermentation science.

---
## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Consistent plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
COLORS = {'ols': '#2196F3', 'ridge': '#4CAF50', 'lasso': '#FF9800', 'elastic': '#9C27B0'}

print('All libraries imported successfully.')

All libraries imported successfully.


---
## 2. Load and Explore the Dataset

The dataset contains 2,000 synthetic observations of artisan cheese fermentation. Each row represents one batch of cheese with its measured biochemical and environmental conditions, and the resulting optimal fermentation time.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
df = pd.read_csv('../data/artisan_cheese_fermentation_data.csv')
print(f'Shape: {df.shape}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'path/to/artisan_cheese_fermentation_data.csv'

In [ ]:
print('Dataset Summary Statistics:')
df.describe().round(2)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTarget variable range: {df.optimal_fermentation_time.min():.1f} to {df.optimal_fermentation_time.max():.1f} hours')

### 2.1 Exploratory Data Analysis (EDA)

Before building any model, we must understand the data. We look at the distribution of the target variable and the correlation between features and the target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
axes[0].hist(df['optimal_fermentation_time'], bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Fermentation Time (Target)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Optimal Fermentation Time (hours)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['optimal_fermentation_time'].mean(), color='red', linestyle='--', label=f'Mean: {df["optimal_fermentation_time"].mean():.1f}h')
axes[0].legend()

# Correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1], square=True, linewidths=0.5, annot_kws={'size': 8})
axes[1].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/fig0_eda.png', bbox_inches='tight')
plt.show()

---
## 3. Data Preprocessing

### 3.1 Why Preprocessing Matters

Linear regression models are sensitive to the scale of features. If `ring_oscillator_speed` is in the hundreds of MHz and `vdd_core` is around 0.9V, the model will incorrectly assign more importance to the larger-scale feature. We use `StandardScaler` to normalize all numeric features to zero mean and unit variance.

**Critical: We fit the scaler ONLY on the training data, then transform both train and test sets.** Fitting on the full dataset would cause data leakage — the model would have indirect knowledge of the test set distribution.

In [ ]:
# Define features and target
TARGET = 'optimal_fermentation_time'
NUMERIC_FEATURES = ['milk_fat_percentage', 'starter_culture_ph', 'ambient_temperature',
                    'fermentation_humidity', 'salt_concentration', 'curd_cut_size', 'aging_room_airflow']
CATEGORICAL_FEATURES = ['bacterial_strain_type']

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

# Train/Validation/Test split: 70% / 15% / 15%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f'Training set:   {X_train.shape[0]} samples')
print(f'Validation set: {X_val.shape[0]} samples')
print(f'Test set:       {X_test.shape[0]} samples')

# Preprocessing pipeline: scale numerics, one-hot encode categoricals
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CATEGORICAL_FEATURES)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f'\nProcessed feature matrix shape: {X_train_proc.shape}')

---
## 4. Ordinary Least Squares (OLS) — From Scratch

### 4.1 The Mathematics

OLS finds the coefficients **w** that minimize the Mean Squared Error (MSE) cost function:

$$J(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \frac{1}{n} \|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2$$

**Closed-Form Solution (Normal Equation):** Setting the gradient to zero gives the exact solution:

$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

This is computationally efficient for small datasets but becomes expensive for very large feature matrices.

In [ ]:
class OLSScratch:
    """Ordinary Least Squares via the Normal Equation."""
    def __init__(self):
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        # Add bias column
        X_b = np.c_[np.ones(X.shape[0]), X]
        # Normal equation: w = (X^T X)^-1 X^T y
        self.theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        self.bias = self.theta[0]
        self.weights = self.theta[1:]
        return self

    def predict(self, X):
        return X @ self.weights + self.bias

ols_scratch = OLSScratch()
ols_scratch.fit(X_train_proc, y_train.values)
y_pred_ols = ols_scratch.predict(X_val_proc)

mse = mean_squared_error(y_val, y_pred_ols)
r2  = r2_score(y_val, y_pred_ols)
print(f'OLS (Normal Equation) — Validation MSE: {mse:.3f} | RMSE: {np.sqrt(mse):.3f} | R²: {r2:.4f}')

### 4.2 Gradient Descent OLS

Gradient Descent is an iterative optimization algorithm. Instead of solving for the exact minimum analytically, it takes small steps in the direction of the steepest descent of the cost function. The update rule is:

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \nabla J(\mathbf{w})$$

where $\alpha$ is the **learning rate** — a hyperparameter controlling the step size. Too large and it overshoots; too small and it converges slowly.

In [ ]:
class OLSGradientDescent:
    """Ordinary Least Squares via Batch Gradient Descent."""
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.cost_history = []
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iter):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y
            # Gradients
            dw = (2 / n_samples) * (X.T @ error)
            db = (2 / n_samples) * np.sum(error)
            # Update
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db
            # Track cost
            cost = np.mean(error ** 2)
            self.cost_history.append(cost)
        return self

    def predict(self, X):
        return X @ self.weights + self.bias

ols_gd = OLSGradientDescent(learning_rate=0.05, n_iterations=2000)
ols_gd.fit(X_train_proc, y_train.values)
y_pred_gd = ols_gd.predict(X_val_proc)

mse_gd = mean_squared_error(y_val, y_pred_gd)
r2_gd  = r2_score(y_val, y_pred_gd)
print(f'OLS (Gradient Descent) — Validation MSE: {mse_gd:.3f} | RMSE: {np.sqrt(mse_gd):.3f} | R²: {r2_gd:.4f}')

# Plot convergence
plt.figure(figsize=(10, 4))
plt.plot(ols_gd.cost_history, color='#2196F3', linewidth=2)
plt.title('Gradient Descent Convergence — Cost vs. Iteration', fontsize=13, fontweight='bold')
plt.xlabel('Iteration')
plt.ylabel('MSE Cost')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/fig_gd_convergence.png', bbox_inches='tight')
plt.show()

---
## 5. Regularization: Ridge, Lasso, and Elastic Net

### Why Regularization?

OLS minimizes only the training error. When a model has many features or correlated features, it can overfit — learning the noise in the training data rather than the true signal. Regularization adds a **penalty term** to the cost function that discourages large coefficients.

| Method | Penalty Added to MSE | Effect |
|:---|:---|:---|
| **Ridge (L2)** | $\alpha \sum w_i^2$ | Shrinks all coefficients toward zero, never exactly zero |
| **Lasso (L1)** | $\alpha \sum |w_i|$ | Drives some coefficients to exactly zero (feature selection) |
| **Elastic Net** | $\alpha_1 \sum |w_i| + \alpha_2 \sum w_i^2$ | Combines both — sparse and stable |

In [ ]:
# Train all three regularized models
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)

models = {'Ridge (L2)': ridge, 'Lasso (L1)': lasso, 'Elastic Net': elastic}
results = {}

for name, model in models.items():
    model.fit(X_train_proc, y_train)
    y_pred = model.predict(X_val_proc)
    mse = mean_squared_error(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2  = r2_score(y_val, y_pred)
    results[name] = {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': mae, 'R2': r2}
    print(f'{name:15s} — RMSE: {np.sqrt(mse):.3f} | MAE: {mae:.3f} | R²: {r2:.4f}')

# Add OLS baseline
ols_sklearn = LinearRegression()
ols_sklearn.fit(X_train_proc, y_train)
y_pred_ols_sk = ols_sklearn.predict(X_val_proc)
mse_sk = mean_squared_error(y_val, y_pred_ols_sk)
results['OLS (sklearn)'] = {'MSE': mse_sk, 'RMSE': np.sqrt(mse_sk),
                             'MAE': mean_absolute_error(y_val, y_pred_ols_sk),
                             'R2': r2_score(y_val, y_pred_ols_sk)}
print(f'{"OLS (sklearn)":15s} — RMSE: {np.sqrt(mse_sk):.3f} | MAE: {mean_absolute_error(y_val, y_pred_ols_sk):.3f} | R²: {r2_score(y_val, y_pred_ols_sk):.4f}')

### 5.1 Coefficient Shrinkage Paths

As the regularization strength (alpha) increases, coefficients are penalized more heavily and shrink toward zero. The key difference: Ridge shrinks them gradually but never to zero, while Lasso can eliminate features entirely.

In [ ]:
alphas = np.logspace(-3, 3, 100)
feature_names = NUMERIC_FEATURES + [f'strain_{i}' for i in range(3)]

ridge_coefs = [Ridge(alpha=a).fit(X_train_proc, y_train).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_train_proc, y_train).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, coef in enumerate(np.array(ridge_coefs).T):
    axes[0].plot(alphas, coef, linewidth=1.5)
axes[0].set_xscale('log')
axes[0].set_title('Ridge (L2) — Coefficient Shrinkage Paths', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Regularization Strength (alpha)')
axes[0].set_ylabel('Coefficient Value')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].grid(True, alpha=0.3)

for i, coef in enumerate(np.array(lasso_coefs).T):
    axes[1].plot(alphas, coef, linewidth=1.5)
axes[1].set_xscale('log')
axes[1].set_title('Lasso (L1) — Coefficient Shrinkage Paths', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Regularization Strength (alpha)')
axes[1].set_ylabel('Coefficient Value')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Coefficient Shrinkage: Ridge vs Lasso', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/fig3_coefficient_paths.png', bbox_inches='tight')
plt.show()
print('Key insight: Lasso drives coefficients to exactly zero (feature selection), Ridge only shrinks them.')

---
## 6. Hyperparameter Tuning — Finding the Best Alpha

The regularization strength `alpha` is a hyperparameter — it is not learned from data, it must be chosen by us. We use **k-fold cross-validation** to find the alpha that minimizes validation error without touching the test set.

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

alphas_cv = np.logspace(-3, 2, 50)

ridge_cv = RidgeCV(alphas=alphas_cv, cv=5)
ridge_cv.fit(X_train_proc, y_train)
print(f'Best Ridge alpha (5-fold CV): {ridge_cv.alpha_:.4f}')

lasso_cv = LassoCV(alphas=alphas_cv, cv=5, max_iter=10000)
lasso_cv.fit(X_train_proc, y_train)
print(f'Best Lasso alpha (5-fold CV): {lasso_cv.alpha_:.4f}')
print(f'Lasso non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)} out of {len(lasso_cv.coef_)}')

---
## 7. Final Evaluation on the Test Set

We evaluate all models on the held-out test set only once — after all hyperparameter decisions are finalized. This gives an unbiased estimate of real-world performance.

In [ ]:
final_models = {
    'OLS (Normal Eq.)': ols_scratch,
    'OLS (sklearn)':    ols_sklearn,
    'Ridge (best alpha)': ridge_cv,
    'Lasso (best alpha)': lasso_cv,
    'Elastic Net':        elastic,
}

print(f'{"Model":<22} {"RMSE":>8} {"MAE":>8} {"R2":>8}')
print('-' * 50)
for name, model in final_models.items():
    if hasattr(model, 'predict'):
        y_pred = model.predict(X_test_proc)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)
        print(f'{name:<22} {rmse:>8.3f} {mae:>8.3f} {r2:>8.4f}')

### 7.1 Residual Analysis

A well-fitted regression model should have residuals (prediction errors) that are randomly distributed around zero with no pattern. Patterns in residuals indicate that the model is missing some structure in the data.

In [ ]:
y_pred_final = ridge_cv.predict(X_test_proc)
residuals = y_test.values - y_pred_final

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_final, alpha=0.4, color='#4CAF50', s=15)
lims = [min(y_test.min(), y_pred_final.min()), max(y_test.max(), y_pred_final.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_title('Predicted vs Actual', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Fermentation Time (h)')
axes[0].set_ylabel('Predicted Fermentation Time (h)')
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(y_pred_final, residuals, alpha=0.4, color='#FF9800', s=15)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Fermentation Time (h)')
axes[1].set_ylabel('Residual (Actual - Predicted)')

# Residual distribution
axes[2].hist(residuals, bins=40, color='#9C27B0', edgecolor='white', alpha=0.85)
axes[2].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Residual Value')
axes[2].set_ylabel('Count')
axes[2].axvline(0, color='red', linestyle='--')

plt.suptitle('Ridge Regression — Residual Analysis (Test Set)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/fig_residuals.png', bbox_inches='tight')
plt.show()

---
## 8. Key Takeaways

1. **OLS (Normal Equation)** provides the exact analytical solution but requires matrix inversion — computationally expensive for large feature sets.
2. **Gradient Descent** is the scalable alternative — it iteratively converges to the minimum and is the foundation of all deep learning optimizers.
3. **Ridge (L2)** is the go-to regularizer when you suspect multicollinearity. It shrinks all coefficients but keeps all features.
4. **Lasso (L1)** performs automatic feature selection by driving irrelevant feature coefficients to exactly zero.
5. **Elastic Net** is the practical choice when you have many correlated features and want both stability and sparsity.
6. **Never tune hyperparameters on the test set.** Use cross-validation on the training set, and reserve the test set for a single final evaluation.

**Next:** Project 002 will apply these same principles to classification using Logistic Regression.